In [ ]:
import os
import json
import re
import sqlite3
import pandas as pd

base_path = "data/raw/Spider/spider_data"

dev_file = os.path.join(base_path, "dev.json")
tables_file = os.path.join(base_path, "tables.json")
pred_file = os.path.join(base_path, "pred_example.txt")

print("Files:")
print("dev.json       :", os.path.exists(dev_file))
print("tables.json    :", os.path.exists(tables_file))
print("pred_examples  :", os.path.exists(pred_file))

Files:
dev.json       : True
tables.json    : True
pred_examples  : True


In [2]:
with open(dev_file, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

print("Number of dev examples:", len(dev_data))

print("\nFirst example:")
print(dev_data[0])

Number of dev examples: 1034

First example:
{'db_id': 'concert_singer', 'query': 'SELECT count(*) FROM singer', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'singer'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'singer'], 'question': 'How many singers do we have?', 'question_toks': ['How', 'many', 'singers', 'do', 'we', 'have', '?'], 'sql': {'from': {'table_units': [['table_unit', 1]], 'conds': []}, 'select': [False, [[3, [0, [0, 0, False], None]]]], 'where': [], 'groupBy': [], 'having': [], 'orderBy': [], 'limit': None, 'intersect': None, 'union': None, 'except': None}}


In [3]:
with open(pred_file, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

predictions = []

for line in raw_lines:
    line = line.strip()

    # Ignore empty lines
    if not line:
        continue

    # Remove markdown-style * characters at beginning/end
    line = line.strip("*").strip()

    if line:
        predictions.append(line)

print("Number of predictions:", len(predictions))

for i, pred in enumerate(predictions, start=1):
    print(i, pred)

Number of predictions: 1034
1 select count(*) from stadium
2 select count(*) from stadium
3 select Age,Name,Country from singer order by Age desc
4 select Age,Name,Country from singer order by Age desc
5 select avg(Age),max(Age),min(Age) from singer where Country = 'terminal'
6 select avg(Age),max(Age),min(Age) from singer where Country = 'terminal'
7 select Song_release_year,Song_Name from singer order by Age asc limit 1
8 select Song_release_year,Song_Name from singer order by Age asc limit 1
9 select Country from singer where Age > 'terminal'
10 select Country from singer where Age > 'terminal'
11 select Country,count(*) from singer group by Country
12 select count(*),Country from singer group by Country
13 select Song_Name from singer where Age > (select avg(Age) from singer)
14 select Song_Name from singer where Age > (select avg(Age) from singer)
15 select Location,Name from stadium where Capacity between 'terminal' and 'terminal'
16 select Location,Name from stadium where Capaci

In [4]:
for i, item in enumerate(dev_data[:10], start=1):
    print("=" * 80)
    print("Example:", i)
    print("DB ID:", item["db_id"])
    print("Question:", item["question"])
    print("Gold SQL:", item["query"])

Example: 1
DB ID: concert_singer
Question: How many singers do we have?
Gold SQL: SELECT count(*) FROM singer
Example: 2
DB ID: concert_singer
Question: What is the total number of singers?
Gold SQL: SELECT count(*) FROM singer
Example: 3
DB ID: concert_singer
Question: Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Example: 4
DB ID: concert_singer
Question: What are the names, countries, and ages for every singer in descending order of age?
Gold SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Example: 5
DB ID: concert_singer
Question: What is the average, minimum, and maximum age of all singers from France?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Example: 6
DB ID: concert_singer
Question: What is the average, minimum, and maximum age for all French singers?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM

In [5]:
def normalize_sql(sql):
    """
    Basic normalization for comparing SQL.
    """
    sql = sql.lower().strip()

    # Remove extra whitespace
    sql = re.sub(r"\s+", " ", sql)

    # Remove trailing semicolon
    sql = sql.rstrip(";").strip()

    return sql


results = []

n = min(len(predictions), len(dev_data))

for i in range(n):
    gold_sql = dev_data[i]["query"]
    pred_sql = predictions[i]

    gold_norm = normalize_sql(gold_sql)
    pred_norm = normalize_sql(pred_sql)

    exact_match = gold_norm == pred_norm

    results.append({
        "index": i + 1,
        "db_id": dev_data[i]["db_id"],
        "question": dev_data[i]["question"],
        "gold_sql": gold_sql,
        "pred_sql": pred_sql,
        "exact_match": exact_match
    })

df = pd.DataFrame(results)

df

,index,db_id,question,gold_sql,pred_sql,exact_match
0,1,concert_singer,How many singers do we have?,SELECT count(*) FROM singer,select count(*) from stadium,False
1,2,concert_singer,What is the total number of singers?,SELECT count(*) FROM singer,select count(*) from stadium,False
2,3,concert_singer,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE...","select Age,Name,Country from singer order by A...",False
3,4,concert_singer,"What are the names, countries, and ages for ev...","SELECT name , country , age FROM singer ORDE...","select Age,Name,Country from singer order by A...",False
4,5,concert_singer,"What is the average, minimum, and maximum age ...","SELECT avg(age) , min(age) , max(age) FROM s...","select avg(Age),max(Age),min(Age) from singer ...",False
...,...,...,...,...,...,...
1029,1030,singer,What are the citizenships that are shared by s...,SELECT Citizenship FROM singer WHERE Birth_Yea...,select Citizenship from singer where Birth_Yea...,False
1030,1031,real_estate_properties,How many available features are there in total?,SELECT count(*) FROM Other_Available_Features,select count(*) from Ref_Feature_Types,False
1031,1032,real_estate_properties,What is the feature type name of feature AirCon?,SELECT T2.feature_type_name FROM Other_Availab...,select T1.feature_type_name from Ref_Feature_T...,False
1032,1033,real_estate_properties,Show the property type descriptions of propert...,SELECT T2.property_type_description FROM Prope...,select property_type_description from Ref_Prop...,False


In [6]:
exact_accuracy = df["exact_match"].mean() * 100

print(f"Exact / Response Accuracy: {exact_accuracy:.2f}%")
print(f"Correct: {df['exact_match'].sum()}")
print(f"Total: {len(df)}")

Exact / Response Accuracy: 12.28%
Correct: 127
Total: 1034


In [7]:
examples_path = os.path.join(
    base_path,
    "evaluation_examples",
    "examples"
)

print("Path exists:", os.path.exists(examples_path))

if os.path.exists(examples_path):
    print("\nContents:")
    for item in os.listdir(examples_path):
        print(item)

Path exists: False


In [ ]:
sqlite_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".sqlite") or file.endswith(".db"):
            sqlite_files.append(os.path.join(root, file))

print("Number of SQLite databases:", len(sqlite_files))

for path in sqlite_files[:20]:
    print(path)

In [9]:
def execute_sql(db_path, sql):
    """
    Execute SQL against SQLite database.
    Returns result or error.
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        cursor.execute(sql)
        result = cursor.fetchall()

        conn.close()

        return {
            "success": True,
            "result": result,
            "error": None
        }

    except Exception as e:
        return {
            "success": False,
            "result": None,
            "error": str(e)
        }

In [10]:
database_path = os.path.join(base_path, "database")

print("Database folder exists:", os.path.exists(database_path))

if os.path.exists(database_path):
    db_ids = os.listdir(database_path)

    print("Number of database folders:", len(db_ids))
    print("First 20:")

    for db in db_ids[:20]:
        print(db)

Database folder exists: True
Number of database folders: 166
First 20:
browser_web
musical
farm
voter_1
game_injury
hospital_1
manufacturer
station_weather
perpetrator
storm_record
flight_1
manufactory_1
cre_Theme_park
museum_visit
race_track
soccer_2
bike_1
pilot_record
customers_and_invoices
department_management


In [11]:
execution_results = []

for i in range(n):

    db_id = dev_data[i]["db_id"]

    gold_sql = dev_data[i]["query"]
    pred_sql = predictions[i]

    db_file = os.path.join(
        database_path,
        db_id,
        f"{db_id}.sqlite"
    )

    if not os.path.exists(db_file):
        execution_results.append({
            "index": i + 1,
            "db_id": db_id,
            "execution_match": False,
            "status": "DATABASE NOT FOUND"
        })
        continue

    gold_result = execute_sql(db_file, gold_sql)
    pred_result = execute_sql(db_file, pred_sql)

    if not gold_result["success"]:
        execution_match = False
        status = "GOLD SQL ERROR"

    elif not pred_result["success"]:
        execution_match = False
        status = "PREDICTION SQL ERROR"

    else:
        execution_match = (
            gold_result["result"] == pred_result["result"]
        )
        status = "OK"

    execution_results.append({
        "index": i + 1,
        "db_id": db_id,
        "gold_sql": gold_sql,
        "pred_sql": pred_sql,
        "gold_result": (
            gold_result["result"]
            if gold_result["success"]
            else gold_result["error"]
        ),
        "pred_result": (
            pred_result["result"]
            if pred_result["success"]
            else pred_result["error"]
        ),
        "execution_match": execution_match,
        "status": status
    })

exec_df = pd.DataFrame(execution_results)

exec_df

,index,db_id,gold_sql,pred_sql,gold_result,pred_result,execution_match,status
0,1,concert_singer,SELECT count(*) FROM singer,select count(*) from stadium,"[(6,)]","[(9,)]",False,OK
1,2,concert_singer,SELECT count(*) FROM singer,select count(*) from stadium,"[(6,)]","[(9,)]",False,OK
2,3,concert_singer,"SELECT name , country , age FROM singer ORDE...","select Age,Name,Country from singer order by A...","[(Joe Sharp, Netherlands, 52), (John Nizinik, ...","[(52, Joe Sharp, Netherlands), (43, John Nizin...",False,OK
3,4,concert_singer,"SELECT name , country , age FROM singer ORDE...","select Age,Name,Country from singer order by A...","[(Joe Sharp, Netherlands, 52), (John Nizinik, ...","[(52, Joe Sharp, Netherlands), (43, John Nizin...",False,OK
4,5,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","select avg(Age),max(Age),min(Age) from singer ...","[(34.5, 25, 43)]","[(None, None, None)]",False,OK
...,...,...,...,...,...,...,...,...
1029,1030,singer,SELECT Citizenship FROM singer WHERE Birth_Yea...,select Citizenship from singer where Birth_Yea...,"[(United States,)]",[],False,OK
1030,1031,real_estate_properties,SELECT count(*) FROM Other_Available_Features,select count(*) from Ref_Feature_Types,"[(3,)]","[(2,)]",False,OK
1031,1032,real_estate_properties,SELECT T2.feature_type_name FROM Other_Availab...,select T1.feature_type_name from Ref_Feature_T...,"[(Amenity, eg Pool.,)]",[],False,OK
1032,1033,real_estate_properties,SELECT T2.property_type_description FROM Prope...,select property_type_description from Ref_Prop...,"[(Apartment, Flat, Condo, etc.,), (Field, Mead...","[(Apartment, Flat, Condo, etc.,), (Field, Mead...",True,OK


In [12]:
execution_accuracy = (
    exec_df["execution_match"].mean() * 100
)

print(f"Execution Accuracy: {execution_accuracy:.2f}%")
print(f"Correct: {exec_df['execution_match'].sum()}")
print(f"Total: {len(exec_df)}")

Execution Accuracy: 33.66%
Correct: 348
Total: 1034


In [13]:
summary = pd.DataFrame({
    "Metric": [
        "Response / Exact Match Accuracy",
        "Execution Accuracy"
    ],
    "Correct": [
        int(df["exact_match"].sum()),
        int(exec_df["execution_match"].sum())
    ],
    "Total": [
        len(df),
        len(exec_df)
    ],
    "Accuracy (%)": [
        round(df["exact_match"].mean() * 100, 2),
        round(exec_df["execution_match"].mean() * 100, 2)
    ]
})

summary

,Metric,Correct,Total,Accuracy (%)
0,Response / Exact Match Accuracy,127,1034,12.28
1,Execution Accuracy,348,1034,33.66
